<a href="https://colab.research.google.com/github/v-yunusic/REPO-PYTHON-VY/blob/main/transacciones-surtidor-2022/bencinera_2022.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CONTEXTO

El conjunto pertenece a datos registrados por nodos IoT durante los mes de enero a marzo 2022.

Se trata de una serie temporal y contiene en general la siguiente información:

  * Nro de la transacción.
  * Fecha y hora de inicio y de fin de la transacción.
  * Cantidad de litros de combustible despachado.
  * Volumen real registrado por otros nodos IoT en los tanques durante el inicio y fin de la descarga.
  * Estación, surtidor, empresa de origen.
  * Tipo de combustible transaccionado.
  * Geolocalización (si es soportada).
  * Unidad de consumo de destino.
  * Sector de la industria a la que pertenece la estación.

Desde la serie más atómica a la agrupación más general, el orden es el siguiente:
id_bomba --> id_equipo --> id_empresa --> id_industria

# QUEHACERES

* Estandarizar columna nombre_prod, definir clases/tipos de combustible en vez de unir nombres alternativos.

# PIP

In [ ]:
pip install matplotlib

In [ ]:
pip install kagglehub

In [1]:
pip install rapidfuzz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 14.9 MB/s  0:00:00


# LIBRERIAS

In [2]:
try:
  import numpy as np
  import matplotlib.pyplot as plt
  import pandas as pd
  import seaborn as sns
  import re # RegEx
  import matplotlib.pyplot as plt
  from sklearn.preprocessing import StandardScaler
  from sklearn.cluster import KMeans
  import gc

  #========= KAGGLE =========#
  import kagglehub

  #======= Rapidfuzz ========#
  from rapidfuzz.distance import Levenshtein

  print('Librerias importadas!')
except Exception as e:
  print(f'Librerias no importadas:\n{e}')

Librerias importadas!


# PREPROCESAMIENTO

In [3]:
# PETICION DE DATA HACIA KAGGLE

path = kagglehub.dataset_download("danielabosch/fuel-transactions-from-gas-pump")
print('DATASET DE KAGGLE CARGADO!')
print(path)

100%|██████████| 21.3M/21.3M [00:00<00:00, 177MB/s]

Extracting files...


DATASET DE KAGGLE CARGADO!
/root/.cache/kagglehub/datasets/danielabosch/fuel-transactions-from-gas-pump/versions/4


In [25]:
data_path = path+'/transacciones_surtidor.csv'

In [29]:
df = pd.read_csv(data_path, sep=',')
df.head()

/tmp/ipykernel_10926/1672446801.py:1: DtypeWarning: Columns (12) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_path, sep=',')


,unique_id_transa,id_transaccion,id_vehiculo,id_usuario,id_equipo,veh_site_id,pump_site_id,tank_site_id,user_site_id,producto,...,temp_final,volumen_comp_15_inicial,volumen_comp_15_final,cantidad_comp_15,codigo_producto,geo_latitud,geo_longitud,id_empresa,id_industria,industria
0,377220101009999,9999,90952,12708,377,000001,1,1,0000,0.0,...,NaN,NaN,NaN,0.000,NaN,NaN,NaN,238,1.0,Transportation
1,504220101006768,6768,101866,6683,504,000001,1,1,0000,0.0,...,NaN,NaN,NaN,158.416,Product 1,NaN,NaN,25,5.0,Distributor
2,534220101007334,7334,123844,34280,534,009593,1,1,3532,0.0,...,30.2,17836.50,17728.5,107.850,0000,NaN,NaN,273,1.0,Transportation
3,878220101002206,2206,141769,46520,878,000103,2,1,0000,0.0,...,NaN,6921.42,NaN,80.002,GOM G3,NaN,NaN,1142,2.0,Mining
4,898220101005224,5224,129356,29688,898,999014,2,1,0001,0.0,...,NaN,NaN,NaN,734.900,Diesel 500,NaN,NaN,1083,1.0,Transportation


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 561747 entries, 0 to 561746
Data columns (total 32 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   unique_id_transa         561747 non-null  int64  
 1   id_transaccion           561747 non-null  int64  
 2   id_vehiculo              561747 non-null  int64  
 3   id_usuario               561747 non-null  int64  
 4   id_equipo                561747 non-null  int64  
 5   veh_site_id              561740 non-null  object 
 6   pump_site_id             561747 non-null  int64  
 7   tank_site_id             561747 non-null  int64  
 8   user_site_id             545281 non-null  object 
 9   producto                 561644 non-null  float64
 10  id_bomba                 561747 non-null  int64  
 11  id_tanque                561747 non-null  int64  
 12  departamento             561747 non-null  object 
 13  cantidad                 561747 non-null  float64
 14  codi

In [7]:
print(f'Cantidad de filas:\n{df.shape[0]}')
print(f'Cantidad de columnas:\n{df.shape[1]}')

Cantidad de filas:
561747
Cantidad de columnas:
32


In [8]:
# DEFINICION DE COLUMNAS NUMERICAS Y CATEGORICAS
col_categoricas = df.select_dtypes(include=['object']).columns.tolist()
col_numericas = df.select_dtypes(exclude=['object','bool']).columns.tolist()
col_bools = df.select_dtypes(include=['bool']).columns.tolist()

In [9]:
nulos = pd.DataFrame({
    "nulos": df.isna().sum(),
    "pct": df.isna().mean() * 100,
}).sort_values("pct", ascending=False)

nulos

,nulos,pct
geo_latitud,555355,98.862121
geo_longitud,555355,98.862121
volumen_final,217475,38.714047
temp_final,217475,38.714047
volumen_comp_15_final,217475,38.714047
codigo_producto,175530,31.247163
temp_inicial,147072,26.181181
volumen_inicial,147072,26.181181
volumen_comp_15_inicial,147072,26.181181
industria,43118,7.675697


In [10]:
df.columns

Index(['unique_id_transa', 'id_transaccion', 'id_vehiculo', 'id_usuario',
       'id_equipo', 'veh_site_id', 'pump_site_id', 'tank_site_id',
       'user_site_id', 'producto', 'id_bomba', 'id_tanque', 'departamento',
       'cantidad', 'codigo_error', 'timestamp', 'timestamp_stop',
       'nombre_prod', 'veh_efficiency', 'volumen_inicial', 'volumen_final',
       'temp_inicial', 'temp_final', 'volumen_comp_15_inicial',
       'volumen_comp_15_final', 'cantidad_comp_15', 'codigo_producto',
       'geo_latitud', 'geo_longitud', 'id_empresa', 'id_industria',
       'industria'],
      dtype='object')

In [11]:
df.head(1)

,unique_id_transa,id_transaccion,id_vehiculo,id_usuario,id_equipo,veh_site_id,pump_site_id,tank_site_id,user_site_id,producto,...,temp_final,volumen_comp_15_inicial,volumen_comp_15_final,cantidad_comp_15,codigo_producto,geo_latitud,geo_longitud,id_empresa,id_industria,industria
0,377220101009999,9999,90952,12708,377,000001,1,1,0000,0.0,...,NaN,NaN,NaN,0.0,NaN,NaN,NaN,238,1.0,Transportation


In [30]:
df = df[[
    'unique_id_transa',
    'producto',
    'id_bomba',
    'cantidad',
    'cantidad_comp_15',
    'codigo_error',
    'timestamp',
    'timestamp_stop',
    'nombre_prod',
    'id_industria',
    'industria'
]]

In [31]:
df.head(2)

,unique_id_transa,producto,id_bomba,cantidad,cantidad_comp_15,codigo_error,timestamp,timestamp_stop,nombre_prod,id_industria,industria
0,377220101009999,0.0,11435,0.000,0.000,UE,2022-01-01 03:49:21,2022-01-01 03:49:21,Producto 0,1.0,Transportation
1,504220101006768,0.0,10980,158.416,158.416,C4,2022-01-01 00:00:36,2022-01-01 00:02:57,Product 1,5.0,Distributor


In [16]:
df.loc[df.duplicated()].shape[0]

0

In [32]:
df['industria'] = df['industria'].fillna('Unknown')
df['nombre_prod'] = df['nombre_prod'].fillna('Unknown')

In [40]:
df[['id_industria','industria']].loc[(df['id_industria'] != 0) & (df['industria']=='Unknown')]

,id_industria,industria
206,NaN,Unknown
425,NaN,Unknown
426,NaN,Unknown
427,NaN,Unknown
435,NaN,Unknown
...,...,...
561343,NaN,Unknown
561344,NaN,Unknown
561345,NaN,Unknown
561346,NaN,Unknown


In [41]:
df['id_industria'] = df['id_industria'].fillna(0)
df['id_industria'].unique()

array([ 1.,  5.,  2.,  4.,  6.,  3.,  0.,  9.,  7.,  8., 10.])

In [42]:
df['industria'].unique()

array(['Transportation', 'Distributor', 'Mining', 'Agriculture',
       'Service Station', 'Oil&Gas', 'Unknown', 'Industry',
       'Construction', 'Public', 'Telcos'], dtype=object)

In [43]:
df.loc[pd.to_datetime(df['timestamp'], errors='coerce').isna()].shape[0], df.loc[pd.to_datetime(df['timestamp_stop'], errors='coerce').isna()].shape[0]

(0, 0)

In [44]:
df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
df['timestamp'] = pd.to_datetime(df['timestamp_stop'], errors='coerce')

In [45]:
df.head(2)

,unique_id_transa,producto,id_bomba,cantidad,cantidad_comp_15,codigo_error,timestamp,timestamp_stop,nombre_prod,id_industria,industria
0,377220101009999,0.0,11435,0.000,0.000,UE,2022-01-01 03:49:21,2022-01-01 03:49:21,Producto 0,1.0,Transportation
1,504220101006768,0.0,10980,158.416,158.416,C4,2022-01-01 00:02:57,2022-01-01 00:02:57,Product 1,5.0,Distributor


# CLASIFICACION POR TIPO COMBUSTIBLE

In [46]:
df.loc[df['nombre_prod'].isna()].shape[0]

0

In [ ]:
len(df['nombre_prod'].unique())

102

In [47]:
df['nombre_prod'].unique()

array(['Producto 0', 'Product 1', 'Diesel 500', 'Gasoil GOM G3',
       'Ultra Diesel', 'VP Nafta', 'Super', 'Gasoil GOM G3 Diesel',
       'Infinia Diesel', 'Nafta Super', 'V-Power Nafta', 'Evolux Diesel',
       'V-Power Diesel', 'Diesel', 'Product 3', 'Gasoil', 'VP Diesel',
       'Diesel Grado 2', 'GASOIL', 'Gas Oil Grado 2', 'Gas Oil',
       'Gasoil Grado 3', 'EURO', 'COMUN', 'GOM', 'GASOIL 2', 'Product 2',
       'Producto 1', 'GAS-OIL - GRADO 3', 'nafta', 'DIESEL', 'VP-Diesel',
       'Formula', 'Urea', 'Infinia', 'Producto 2', 'GAS OIL ',
       'Gasoil GOM', 'INFINIA DIESEL', 'GAS OIL', 'Gasoil Comun',
       'Euro Diesel', 'DIESEL-500', 'Ultra Diesel ', 'DIESEL 1',
       'COMBUSTIBLE -  DIESEL 500', 'Gasoil Infinia', 'Diesel Premium',
       'Diesel Comun ', 'Axion Diesel', 'ULTRADIESEL', 'EURODIESEL',
       'Euro-Diesel', 'Ultra-Diesel', 'Nafta', 'Gasoil 500',
       'Diesel 500 PPM', 'S1002050010', 'Puerto Seco La Puerta SA',
       'Cereales Norte SA', 'Gas Oil Comun Gr

In [138]:
productos = [p.strip(' \t\n\r').upper() for p in df['nombre_prod'].unique() if p]
productos = list(set(productos))

combustibles = pd.DataFrame(columns=['Producto','Tipo'])
combustibles['Producto'] = productos
combustibles.head(3)

,Producto,Tipo
0,UD,NaN
1,V-POWER NAFTA,NaN
2,GAS OIL,NaN


In [139]:
combustibles.shape[0]

83

In [140]:
# TIPOS DE PRODUCTO:

tipo_diesel = ['infinia','diesel','diésel','euro','product',
               'producto','combustible','gasoil','gas oil','gom',
               'gas-oil','ultra','formula']

tipo_nafta = ['nafta','super']

tipo_urea = ['urea','azul']

for p in combustibles['Producto']:
  if isinstance(p, str):
    producto = p.lower().strip()

    if any(t in producto for t in tipo_diesel):
      combustibles.loc[combustibles['Producto'].str.lower().str.strip() == producto, 'Tipo'] = 'Diesel'

    if any(t in producto for t in tipo_nafta):
      combustibles.loc[combustibles['Producto'].str.lower().str.strip() == producto, 'Tipo'] = 'Nafta'

    if any(t in producto for t in tipo_urea):
      combustibles.loc[combustibles['Producto'].str.lower().str.strip() == producto, 'Tipo'] = 'Urea'

    combustibles['Tipo'] = combustibles['Tipo'].fillna('Otro')



In [147]:
combustibles.loc[combustibles['Tipo'] == 'Diesel']

,Producto,Tipo
2,GAS OIL,Diesel
5,GASOIL 1,Diesel
6,DIESEL 500 G2,Diesel
7,AXION DIESEL,Diesel
8,DIESEL 500,Diesel
...,...,...
78,PRODUCTO 1 - EURO DIESEL,Diesel
79,DIESEL GRADO 2,Diesel
80,DIESEL 500 PPM,Diesel
81,COMBUSTIBLE,Diesel


In [142]:
combustibles['Tipo'].unique()

array(['Otro', 'Nafta', 'Diesel', 'Urea'], dtype=object)

In [148]:
dict(zip(combustibles.Producto, combustibles.Tipo))

{'UD': 'Otro',
 'V-POWER NAFTA': 'Nafta',
 'GAS OIL': 'Diesel',
 'NAFTA': 'Nafta',
 'UREA': 'Urea',
 'GASOIL 1': 'Diesel',
 'DIESEL 500 G2': 'Diesel',
 'AXION DIESEL': 'Diesel',
 'DIESEL 500': 'Diesel',
 'DIESEL GRADO 3': 'Diesel',
 'INFINIA DIESEL': 'Diesel',
 'EURO G3': 'Diesel',
 'INFINIA DIÉSEL': 'Diesel',
 'ULTRA DIESEL': 'Diesel',
 'S1002050010': 'Otro',
 'GAS-OIL': 'Diesel',
 'PUERTO SECO LA PUERTA SA': 'Otro',
 'SUPER': 'Nafta',
 'GASOIL GOM G3 DIESEL': 'Diesel',
 'COMBUSTIBLE -  DIESEL 500': 'Diesel',
 'EURO DIESEL GRADO3': 'Diesel',
 'PRODUCTO 1- EURO DIESEL': 'Diesel',
 'INFINIA': 'Diesel',
 'DIESEL PREMIUM': 'Diesel',
 'GASOIL INFINIA': 'Diesel',
 'GAS OIL COMUN GR 2': 'Diesel',
 'GAS OIL GRADO 2': 'Diesel',
 'GASOIL GRADO 3': 'Diesel',
 'VP-DIESEL': 'Diesel',
 'ULTRADIESEL': 'Diesel',
 'GAS OIL 500': 'Diesel',
 'PRODUCT 3': 'Diesel',
 'GASOIL 2': 'Diesel',
 'CEREALES NORTE SA': 'Otro',
 'PRODUCTO 2 - DIESEL': 'Diesel',
 'AZUL 32': 'Urea',
 'GASOIL G3': 'Diesel',
 'PRODUCTO